In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")

tz = pytz.timezone('Asia/Shanghai')

def to_tz_datetime(s):
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

# --- Metrics aggregation ---
metrics_grp = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    count=('value', 'count'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    mean_value=('value', 'mean'),
    median_value=('value', 'median'),
    p95_value=('value', lambda x: np.percentile(x, 95))
)

# convert timestamps to Asia/Shanghai timezone-aware datetimes
metrics_grp['min_timestamp'] = to_tz_datetime(metrics_grp['min_timestamp'])
metrics_grp['max_timestamp'] = to_tz_datetime(metrics_grp['max_timestamp'])

# round numeric columns for compact display
metrics_grp['mean_value'] = metrics_grp['mean_value'].round(6)
metrics_grp['median_value'] = metrics_grp['median_value'].round(6)
metrics_grp['p95_value'] = metrics_grp['p95_value'].round(6)

# reorder and limit to top 20 by count
metrics_summary = metrics_grp[['cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'max_timestamp', 'mean_value', 'median_value', 'p95_value']]
metrics_summary = metrics_summary.sort_values('count', ascending=False).head(20)

# --- Trace aggregation ---
traces_grp = df_trace.groupby('trace_name', as_index=False).agg(
    count=('value', 'count'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max'),
    mean_value=('value', 'mean'),
    median_value=('value', 'median'),
    p95_value=('value', lambda x: np.percentile(x, 95))
)

traces_grp['min_timestamp'] = to_tz_datetime(traces_grp['min_timestamp'])
traces_grp['max_timestamp'] = to_tz_datetime(traces_grp['max_timestamp'])

traces_grp['mean_value'] = traces_grp['mean_value'].round(6)
traces_grp['median_value'] = traces_grp['median_value'].round(6)
traces_grp['p95_value'] = traces_grp['p95_value'].round(6)

traces_summary = traces_grp[['trace_name', 'count', 'min_timestamp', 'max_timestamp', 'mean_value', 'median_value', 'p95_value']]

# If many trace rows, keep top 20 by count for compactness
if traces_summary.shape[0] > 20:
    traces_summary = traces_summary.sort_values('count', ascending=False).head(20)

# Display compact summaries (variables as outputs)
metrics_summary, traces_summary

```
Out[1]:
```
```python
# Summarize the previously computed aggregates in plain English.
# Reuse existing variables: metrics_summary, traces_summary

n_metrics = metrics_summary.shape[0]
metrics_min_ts = metrics_summary['min_timestamp'].min()
metrics_max_ts = metrics_summary['max_timestamp'].max()
most_common_count_metric = metrics_summary['count'].mode()[0]

top_metrics_rows = metrics_summary.sort_values(['count','mean_value'], ascending=[False, False]).head(5)
top_metrics_lines = []
for _, r in top_metrics_rows.iterrows():
    top_metrics_lines.append(
        f"{r['cmdb_id']} / {r['kpi_name']}: count={int(r['count'])}, mean={r['mean_value']}, "
        f"median={r['median_value']}, p95={r['p95_value']}"
    )

n_traces = traces_summary.shape[0]
traces_min_ts = traces_summary['min_timestamp'].min()
traces_max_ts = traces_summary['max_timestamp'].max()
most_common_count_trace = traces_summary['count'].mode()[0]

top_traces_rows = traces_summary.sort_values(['count','mean_value'], ascending=[False, False]).head(5)
top_traces_lines = []
for _, r in top_traces_rows.iterrows():
    top_traces_lines.append(
        f"{r['trace_name']}: count={int(r['count'])}, mean={r['mean_value']}, "
        f"median={r['median_value']}, p95={r['p95_value']}"
    )

summary = (
    "Summary of aggregated telemetry:\n\n"
    f"Metrics: showing top {n_metrics} metric series by point count.\n"
    f"- Time range (metrics displayed): {metrics_min_ts} to {metrics_max_ts} (Asia/Shanghai).\n"
    f"- Most metric series have count = {most_common_count_metric} (several with 61 points, some with 60).\n"
    "- Top metric series (examples):\n  - " + "\n  - ".join(top_metrics_lines) + "\n\n"
    f"Traces: showing {n_traces} trace series (top by count if many).\n"
    f"- Time range (traces displayed): {traces_min_ts} to {traces_max_ts} (Asia/Shanghai).\n"
    f"- Most common trace series count = {most_common_count_trace} (examples below).\n"
    "- Top trace series (examples):\n  - " + "\n  - ".join(top_traces_lines) + "\n\n"
    "Notes: outputs are compact aggregates (count, min/max timestamps, mean, median, p95)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id                  kpi_name  count             min_timestamp             max_timestamp  mean_value  median_value   p95_value
1782  osb_001                   app.num     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00  411.278689    418.000000  465.000000
5      db_001              CPU_free_pct     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00   98.367009     98.611035   98.997902
955    os_006                Send_total     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.000370      0.000366    0.000474
95     db_003              CPU_free_pct     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00   98.814997     98.972677   99.348405
289    db_007             MEM_real_util     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00   82.147213     82.150000   82.160000
1395   os_015  Incoming_network_traffic     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.002212      0.002300    0.002379
1516   os_017                Send_total     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.216835      0.224460    0.251811
1761   os_022  Outgoing_network_traffic     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.059520      0.058163    0.066095
976    os_007               Disk_rd_ios     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.000000      0.000000    0.000000
1784  osb_001           app.succee_rate     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.999496      1.000000    1.000000
1781  osb_001              app.avg_time     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.002937      0.000545    0.005620
1057   os_008                Send_total     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.000580      0.000563    0.000681
1230   os_012              Disk_io_util     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.585376      0.500000    1.058012
1256   os_012      Processor_load_5_min     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.011612      0.011667    0.020000
1638   os_020              Disk_io_util     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00    0.000000      0.000000    0.000000
1783  osb_001            app.succee_num     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00  411.131148    418.000000  465.000000
322    db_008              CPU_free_pct     61 2020-05-28 03:00:00+08:00 2020-05-28 04:00:00+08:00   97.070576     97.138207   97.638150
816    os_004           CPU_system_time     60 2020-05-28 03:00:00+08:00 2020-05-28 03:59:00+08:00    0.577701      0.575607    0.649782
802    os_003                Send_total     60 2020-05-28 03:00:00+08:00 2020-05-28 03:59:00+08:00    3.452268      3.500890    3.597767
1627   os_019                  ss_total     60 2020-05-28 03:00:00+08:00 2020-05-28 03:59:00+08:00  357.366667    352.000000  405.250000,                                            trace_name  count             min_timestamp             max_timestamp   mean_value  median_value    p95_value
55                         trace.self.local.row_count    240 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00   918.695833    657.500000  1596.000000
54                        trace.self.local.error_rate    240 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.000087      0.000000     0.000000
53                      trace.self.local.duration_p95    240 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.070150      0.018000     0.097000
52                     trace.self.local.duration_mean    240 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.019869      0.007782     0.019569
51                          trace.self.jdbc.row_count    240 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00  1194.845833   1089.000000  1826.050000
50                         trace.self.jdbc.error_rate    240 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.000000      0.000000     0.000000
49                       trace.self.jdbc.duration_p95    240 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.034001      0.003000     0.008150
48                      trace.self.jdbc.duration_mean    240 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.010853      0.002593     0.007268
43                           trace.self.csf.row_count    180 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00   306.350000    358.000000   468.000000
42                          trace.self.csf.error_rate    180 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.000282      0.000000     0.000000
41                        trace.self.csf.duration_p95    180 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     1.425462      0.184750     6.545125
40                       trace.self.csf.duration_mean    180 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.472965      0.074654     2.853495
47                     trace.self.flyremote.row_count    120 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00    91.925000     99.500000   118.050000
46                    trace.self.flyremote.error_rate    120 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.000000      0.000000     0.000000
45                  trace.self.flyremote.duration_p95    120 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.053658      0.009175     0.095615
44                 trace.self.flyremote.duration_mean    120 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.013998      0.007421     0.017957
0   trace.from_docker_001.remoteprocess.duration_mean     60 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.045581      0.045696     0.047048
1    trace.from_docker_001.remoteprocess.duration_p95     60 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.054085      0.054000     0.056372
2      trace.from_docker_001.remoteprocess.error_rate     60 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00     0.000000      0.000000     0.000000
3       trace.from_docker_001.remoteprocess.row_count     60 2020-05-28 03:30:00+08:00 2020-05-28 03:59:00+08:00   182.733333    200.000000   230.150000)```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing variables: df_metric, metrics_summary (global aggregates from prior run)
# If not present, load df_metric (but per instructions they exist)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# 1) Compute global P5 for each (cmdb_id, kpi_name) using the full metric series
p5_series = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.05)
p5_series = p5_series.rename(columns={'value': 'p5_value'})

# Attach p5_value to existing metrics_summary (left join so we keep the previously computed rows)
metrics_with_p5 = metrics_summary.merge(p5_series, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Filter metric.csv to the incident window (Asia/Shanghai timezone)
tz = pytz.timezone('Asia/Shanghai')
start_ts = int(pd.Timestamp('2020-05-28 03:30:00', tz=tz).tz_convert('UTC').timestamp())
end_ts = int(pd.Timestamp('2020-05-28 04:00:00', tz=tz).tz_convert('UTC').timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) For each (cmdb_id, kpi_name) within the window compute anomaly stats using the global p5/p95
# Merge p5/p95 into window rows
df_window = df_window.merge(metrics_with_p5[['cmdb_id', 'kpi_name', 'p5_value', 'p95_value']],
                            on=['cmdb_id', 'kpi_name'], how='left')

# Determine anomalies (guard against NaN thresholds)
df_window['high_anom'] = (df_window['value'] > df_window['p95_value']) & df_window['p95_value'].notna()
df_window['low_anom']  = (df_window['value'] < df_window['p5_value'])  & df_window['p5_value'].notna()

# Earliest anomaly timestamp helper column (numeric epoch seconds), NaN if not anomaly
df_window['anomalous_ts'] = np.where(df_window['high_anom'] | df_window['low_anom'],
                                    df_window['timestamp'], np.nan)

# Deviation percentages (handle division by zero)
df_window['high_dev_pct'] = np.where(
    df_window['high_anom'] & (df_window['p95_value'] != 0),
    (df_window['value'] - df_window['p95_value']) / df_window['p95_value'] * 100.0,
    np.nan
)
df_window['low_dev_pct'] = np.where(
    df_window['low_anom'] & (df_window['p5_value'] != 0),
    (df_window['p5_value'] - df_window['value']) / df_window['p5_value'] * 100.0,
    np.nan
)

# Aggregate per (cmdb_id, kpi_name)
agg = df_window.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    anomaly_high_count = ('high_anom', 'sum'),
    anomaly_low_count  = ('low_anom', 'sum'),
    earliest_anomaly_ts = ('anomalous_ts', 'min'),
    max_high_deviation_pct = ('high_dev_pct', 'max'),
    max_low_deviation_pct  = ('low_dev_pct', 'max')
)

# Ensure counts are integers
agg['anomaly_high_count'] = agg['anomaly_high_count'].astype(int)
agg['anomaly_low_count'] = agg['anomaly_low_count'].astype(int)

# Convert earliest_anomaly_ts to Asia/Shanghai timezone-aware datetimes (NULL if NaN)
def to_tz_datetime_series(s):
    s = pd.to_numeric(s, errors='coerce')
    return pd.to_datetime(s, unit='s', utc=True).dt.tz_convert(tz)

agg['earliest_anomaly_ts'] = to_tz_datetime_series(agg['earliest_anomaly_ts'])

# Merge back p5/p95 for display
result = agg.merge(metrics_with_p5[['cmdb_id', 'kpi_name', 'p5_value', 'p95_value']],
                   on=['cmdb_id', 'kpi_name'], how='left')

# Filter to only those with any anomalies
result['total_anomalies'] = result['anomaly_high_count'] + result['anomaly_low_count']
result = result[result['total_anomalies'] > 0].copy()

# Sort by total anomalies desc and keep top 20 rows
result = result.sort_values('total_anomalies', ascending=False).head(20)

# Round deviation percentages for compactness
result['max_high_deviation_pct'] = result['max_high_deviation_pct'].round(3)
result['max_low_deviation_pct'] = result['max_low_deviation_pct'].round(3)

# Select and order requested columns
final_cols = ['cmdb_id', 'kpi_name', 'p5_value', 'p95_value',
              'anomaly_high_count', 'anomaly_low_count', 'earliest_anomaly_ts',
              'max_high_deviation_pct', 'max_low_deviation_pct']

final_result = result[final_cols]

# Display compact final table
final_result

```
Out[2]:
```
```python
# Summarize the anomaly analysis results in plain English using existing variable final_result.
# Reuse final_result from the kernel and produce a compact summary string.

try:
    final_result
except NameError:
    import pandas as pd
    final_result = pd.DataFrame()  # fallback (should not happen in this session)

n_affected = final_result.shape[0]
time_window = "2020-05-28 03:30:00+08:00 to 2020-05-28 04:00:00+08:00"
# compute total anomalies for sorting
final_result['total_anomalies'] = final_result['anomaly_high_count'] + final_result['anomaly_low_count']
top_rows = final_result.sort_values('total_anomalies', ascending=False).head(5)

lines = [
    f"Anomaly summary for window {time_window} (Asia/Shanghai):",
    f"- {n_affected} component-KPI series had at least one anomaly (value beyond global P5/P95).",
    "- Top 5 affected component-KPIs (cmdb_id / kpi_name):"
]

for _, r in top_rows.iterrows():
    lines.append(
        f"  • {r['cmdb_id']} / {r['kpi_name']}: total_anomalies={int(r['total_anomalies'])} "
        f"(high={int(r['anomaly_high_count'])}, low={int(r['anomaly_low_count'])}), "
        f"earliest={r['earliest_anomaly_ts']}, p5={r['p5_value']}, p95={r['p95_value']}, "
        f"max_high_dev%={r['max_high_deviation_pct'] if pd.notna(r['max_high_deviation_pct']) else 'NULL'}, "
        f"max_low_dev%={r['max_low_deviation_pct'] if pd.notna(r['max_low_deviation_pct']) else 'NULL'}"
    )

# Highlight particularly severe cases
severe = top_rows.iloc[0]
lines.append(
    f"- Notable: {severe['cmdb_id']} / {severe['kpi_name']} shows the largest anomaly impact "
    f"with total_anomalies={int(severe['total_anomalies'])} and very large max_high_deviation_pct="
    f"{severe['max_high_deviation_pct']}% (early anomaly at {severe['earliest_anomaly_ts']})."
)

lines.append("- End of summary. Detailed compact table is available in the prior result variable.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                  kpi_name    p5_value   p95_value  anomaly_high_count  anomaly_low_count       earliest_anomaly_ts  max_high_deviation_pct  max_low_deviation_pct
1753  osb_001              app.avg_time    0.000469    0.005620                   3                  3 2020-05-28 03:43:00+08:00                1107.506                  8.513
5      db_001              CPU_free_pct   97.143088   98.997902                   3                  2 2020-05-28 03:32:00+08:00                   0.101                  7.615
1040   os_008                Send_total    0.000517    0.000681                   2                  3 2020-05-28 03:30:00+08:00                   0.294                  0.967
1754  osb_001                   app.num   86.000000  465.000000                   1                  3 2020-05-28 03:48:00+08:00                 107.742                 23.256
91     db_003              CPU_free_pct   97.799505   99.348405                   3                  1 2020-05-28 03:31:00+08:00                   0.159                  1.593
1755  osb_001            app.succee_num   83.500000  465.000000                   1                  3 2020-05-28 03:48:00+08:00                 107.097                 20.958
1492   os_017                Send_total    0.091123    0.251811                   0                  3 2020-05-28 03:49:00+08:00                     NaN                 27.678
938    os_006                Send_total    0.000308    0.000474                   1                  2 2020-05-28 03:33:00+08:00                   0.422                  0.649
312    db_008              CPU_free_pct   96.295133   97.638150                   2                  1 2020-05-28 03:37:00+08:00                   0.081                  0.842
1733   os_022  Outgoing_network_traffic    0.028629    0.066095                   0                  3 2020-05-28 03:48:00+08:00                     NaN                 29.694
1756  osb_001           app.succee_rate    0.996900    1.000000                   0                  3 2020-05-28 03:47:00+08:00                     NaN                  1.184
786    os_003                Send_total    3.059166    3.597767                   1                  1 2020-05-28 03:46:00+08:00                   2.792                  1.239
1212   os_012              Disk_io_util    0.000000    1.058012                   2                  0 2020-05-28 03:37:00+08:00                  89.034                    NaN
1601   os_019                  ss_total  326.850000  405.250000                   2                  0 2020-05-28 03:41:00+08:00                   1.912                    NaN
1237   os_012      Processor_load_5_min    0.003333    0.020000                   1                  1 2020-05-28 03:31:00+08:00                  25.000                 49.985
1374   os_015  Incoming_network_traffic    0.000000    0.002379                   2                  0 2020-05-28 03:31:00+08:00                  46.795                    NaN
800    os_004           CPU_system_time    0.497091    0.649782                   0                  1 2020-05-28 03:59:00+08:00                     NaN                  0.419```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables from kernel: df_trace, traces_summary
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    traces_summary
except NameError:
    # If traces_summary not present, compute minimal p95 from full series (but instruction says it's available)
    traces_summary = df_trace.groupby('trace_name', as_index=False).agg(
        p95_value=('value', lambda x: np.percentile(x, 95))
    )

tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P5 for each trace_name (from full series) and attach p95 from traces_summary
p5_df = df_trace.groupby('trace_name', as_index=False)['value'].quantile(0.05).rename(columns={'value': 'p5_value'})

# Ensure traces_summary has trace_name and p95_value
if 'p95_value' not in traces_summary.columns:
    traces_summary = traces_summary.rename(columns={traces_summary.columns[-1]: 'p95_value'})

trace_thresholds = traces_summary[['trace_name', 'p95_value']].merge(p5_df, on='trace_name', how='left')

# 2) Filter trace.csv to the incident window (inclusive)
start_ts = int(pd.Timestamp('2020-05-28 03:30:00', tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp('2020-05-28 04:00:00', tz=tz).tz_convert('UTC').timestamp())

df_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) For each trace_name in window compute anomaly stats using global thresholds
# Merge thresholds into window
df_window = df_window.merge(trace_thresholds, on='trace_name', how='left')

# Identify anomalies (guard against missing thresholds)
df_window['high_anom'] = df_window['p95_value'].notna() & (df_window['value'] > df_window['p95_value'])
df_window['low_anom']  = df_window['p5_value'].notna()  & (df_window['value'] < df_window['p5_value'])

# earliest anomaly ts helper (epoch seconds)
df_window['anomalous_ts'] = np.where(df_window['high_anom'] | df_window['low_anom'],
                                    df_window['timestamp'], np.nan)

# deviation percentages (avoid divide by zero)
df_window['high_dev_pct'] = np.where(
    df_window['high_anom'] & (df_window['p95_value'] != 0),
    (df_window['value'] - df_window['p95_value']) / df_window['p95_value'] * 100.0,
    np.nan
)
df_window['low_dev_pct'] = np.where(
    df_window['low_anom'] & (df_window['p5_value'] != 0),
    (df_window['p5_value'] - df_window['value']) / df_window['p5_value'] * 100.0,
    np.nan
)

# Aggregate per trace_name
agg = df_window.groupby('trace_name', as_index=False).agg(
    anomaly_high_count = ('high_anom', 'sum'),
    anomaly_low_count  = ('low_anom', 'sum'),
    earliest_anomaly_ts = ('anomalous_ts', 'min'),
    max_high_deviation_pct = ('high_dev_pct', 'max'),
    max_low_deviation_pct  = ('low_dev_pct', 'max')
)

# Cast counts to int
agg['anomaly_high_count'] = agg['anomaly_high_count'].astype(int)
agg['anomaly_low_count']  = agg['anomaly_low_count'].astype(int)

# Convert earliest_anomaly_ts to timezone-aware datetimes (Asia/Shanghai), NULL if NaN
agg['earliest_anomaly_ts'] = pd.to_numeric(agg['earliest_anomaly_ts'], errors='coerce')
agg['earliest_anomaly_ts'] = pd.to_datetime(agg['earliest_anomaly_ts'], unit='s', utc=True).dt.tz_convert(tz)

# Merge thresholds (p5/p95) for display
result = agg.merge(trace_thresholds[['trace_name', 'p5_value', 'p95_value']], on='trace_name', how='left')

# Filter to only those with any anomalies and sort by total anomalies desc
result['total_anomalies'] = result['anomaly_high_count'] + result['anomaly_low_count']
result = result[result['total_anomalies'] > 0].copy()
result = result.sort_values('total_anomalies', ascending=False).head(20)

# Round deviation percentages for compactness
result['max_high_deviation_pct'] = result['max_high_deviation_pct'].round(3)
result['max_low_deviation_pct']  = result['max_low_deviation_pct'].round(3)

# Select requested columns in order
final_trace_anomalies = result[['trace_name', 'p5_value', 'p95_value',
                                'anomaly_high_count', 'anomaly_low_count',
                                'earliest_anomaly_ts', 'max_high_deviation_pct', 'max_low_deviation_pct']]

# Display compact final table variable
final_trace_anomalies

```
Out[3]:
```
```python
# Summarize trace anomaly analysis in plain English using existing variable final_trace_anomalies.
# Output a concise human-readable summary string.

try:
    df = final_trace_anomalies.copy()
except NameError:
    import pandas as pd
    df = pd.DataFrame()  # fallback (should not happen)

# compute totals and top entries
if df.empty:
    summary = "No trace anomaly data available."
else:
    df['total_anomalies'] = df['anomaly_high_count'] + df['anomaly_low_count']
    n_traces = int(df.shape[0])
    time_window = "2020-05-28 03:30:00+08:00 through 2020-05-28 04:00:00+08:00"
    top5 = df.sort_values('total_anomalies', ascending=False).head(5)

    lines = [
        f"Trace anomaly summary for window {time_window} (Asia/Shanghai):",
        f"- {n_traces} trace metrics had anomalies (value outside global P5/P95).",
        "- Top 5 affected traces (trace_name: total | high, low anomalies; earliest anomaly; p5/p95; max high% / max low%):"
    ]

    for _, r in top5.iterrows():
        lines.append(
            f"  • {r['trace_name']}: {int(r['total_anomalies'])} "
            f"({int(r['anomaly_high_count'])} high, {int(r['anomaly_low_count'])} low); "
            f"earliest={r['earliest_anomaly_ts']}; p5={r['p5_value']}; p95={r['p95_value']}; "
            f"max_high_dev%={r['max_high_deviation_pct'] if pd.notna(r['max_high_deviation_pct']) else 'NULL'}; "
            f"max_low_dev%={r['max_low_deviation_pct'] if pd.notna(r['max_low_deviation_pct']) else 'NULL'}"
        )

    # note most severe by total anomalies (first row) and by largest high deviation
    most_total = top5.iloc[0]
    # largest high deviation overall
    idx_max_high = df['max_high_deviation_pct'].replace([float('inf'), -float('inf')], pd.NA).idxmax()
    if pd.notna(idx_max_high):
        most_severe_high = df.loc[idx_max_high]
        severe_note = (
            f"- Most anomalous by count: {most_total['trace_name']} with {int(most_total['total_anomalies'])} anomalies, "
            f"earliest at {most_total['earliest_anomaly_ts']}."
        )
        severe_high_note = (
            f"- Largest high deviation observed on {most_severe_high['trace_name']}: "
            f"max_high_dev%={most_severe_high['max_high_deviation_pct']}%."
        )
        lines.append(severe_note)
        lines.append(severe_high_note)
    else:
        lines.append(f"- Most anomalous by count: {most_total['trace_name']} with {int(most_total['total_anomalies'])} anomalies.")

    lines.append("- End of summary. See compact table variable for details.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

trace_name    p5_value    p95_value  anomaly_high_count  anomaly_low_count       earliest_anomaly_ts  max_high_deviation_pct  max_low_deviation_pct
51                          trace.self.jdbc.row_count  414.800000  1826.050000                  12                 12 2020-05-28 03:36:00+08:00                  11.388                 28.640
48                      trace.self.jdbc.duration_mean    0.001493     0.007268                  12                 12 2020-05-28 03:37:00+08:00                3611.324                 30.690
52                     trace.self.local.duration_mean    0.004502     0.019569                  12                 12 2020-05-28 03:33:00+08:00                2012.964                 33.410
55                         trace.self.local.row_count  248.250000  1596.000000                  11                 12 2020-05-28 03:36:00+08:00                  11.404                 28.298
53                      trace.self.local.duration_p95    0.007950     0.097000                  10                 12 2020-05-28 03:33:00+08:00                1563.711                 24.528
40                       trace.self.csf.duration_mean    0.036366     2.853495                   9                  9 2020-05-28 03:33:00+08:00                 149.923                  6.878
41                        trace.self.csf.duration_p95    0.085995     6.545125                   9                  9 2020-05-28 03:35:00+08:00                 203.637                 32.554
43                           trace.self.csf.row_count   95.900000   468.000000                   8                  9 2020-05-28 03:36:00+08:00                   8.547                 32.221
44                 trace.self.flyremote.duration_mean    0.005279     0.017957                   6                  6 2020-05-28 03:43:00+08:00                1614.897                  5.283
49                       trace.self.jdbc.duration_p95    0.003000     0.008150                  12                  0 2020-05-28 03:39:00+08:00               11299.387                    NaN
47                     trace.self.flyremote.row_count   34.950000   118.050000                   6                  6 2020-05-28 03:48:00+08:00                   7.582                 19.886
45                  trace.self.flyremote.duration_p95    0.006797     0.095615                   6                  6 2020-05-28 03:47:00+08:00                1518.731                 11.732
0   trace.from_docker_001.remoteprocess.duration_mean    0.044075     0.047048                   3                  3 2020-05-28 03:30:00+08:00                   0.848                  2.256
3       trace.from_docker_001.remoteprocess.row_count   62.750000   230.150000                   3                  3 2020-05-28 03:46:00+08:00                  12.101                 15.538
1    trace.from_docker_001.remoteprocess.duration_p95    0.052000     0.056372                   3                  1 2020-05-28 03:35:00+08:00                   1.114                  1.154
42                          trace.self.csf.error_rate    0.000000     0.000000                   4                  0 2020-05-28 03:47:00+08:00                     NaN                    NaN
54                        trace.self.local.error_rate    0.000000     0.000000                   2                  0 2020-05-28 03:47:00+08:00                     NaN                    NaN```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables from kernel: df_metric, metrics_summary
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    metrics_summary
except NameError:
    # Fallback: compute p95 from full series if metrics_summary missing (still global, not from window)
    metrics_summary = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        p95_value=('value', lambda x: np.percentile(x, 95))
    )

tz = pytz.timezone('Asia/Shanghai')

# Define target component and KPI name set (app.*)
cmdb_target = 'osb_001'
kpi_prefix = 'app.'
kpi_set = ['app.avg_time', 'app.num', 'app.succee_rate', 'app.succee_num']

# 1) Obtain global p95_value from metrics_summary and compute global p5_value from full series
# Ensure metrics_summary contains p95_value
if 'p95_value' not in metrics_summary.columns:
    # attempt to rename if last column is p95
    metrics_summary = metrics_summary.rename(columns={metrics_summary.columns[-1]: 'p95_value'})

p5_df = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False)['value'].quantile(0.05).rename(columns={'value': 'p5_value'})

thresholds = metrics_summary[['cmdb_id', 'kpi_name', 'p95_value']].merge(p5_df, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Filter metric.csv to incident window and selected component/KPIs
start_ts = int(pd.Timestamp('2020-05-28 03:30:00', tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp('2020-05-28 04:00:00', tz=tz).tz_convert('UTC').timestamp())

df_window = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'] == cmdb_target) &
    (df_metric['kpi_name'].isin(kpi_set))
].copy()

# 3) Merge thresholds (global) and mark anomalies (do NOT recompute thresholds from window)
df_window = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomaly types; ignore rows where thresholds are missing (will result in NaN and no anomaly)
df_window['anomaly_type'] = np.where(
    df_window['value'] > df_window['p95_value'], 'high',
    np.where(df_window['value'] < df_window['p5_value'], 'low', pd.NA)
)

# Keep only anomalous points
df_anom = df_window[df_window['anomaly_type'].notna()].copy()

# 4) Group consecutive anomalies of same type into contiguous-by-60s segments
if not df_anom.empty:
    # Sort for grouping
    df_anom = df_anom.sort_values(['cmdb_id', 'kpi_name', 'anomaly_type', 'timestamp']).reset_index(drop=True)

    # For each group (cmdb_id, kpi_name, anomaly_type) compute segment ids where timestamp gap != 60
    def assign_segment_ids(g):
        g = g.sort_values('timestamp').copy()
        # compute gap (first row starts new segment)
        g['gap'] = g['timestamp'].diff().fillna(9999)  # large for first row
        g['new_segment'] = (g['gap'] != 60).astype(int)
        g['segment_id'] = g['new_segment'].cumsum()  # relative to this group's order
        return g

    df_segments = df_anom.groupby(['cmdb_id', 'kpi_name', 'anomaly_type'], group_keys=False).apply(assign_segment_ids).reset_index(drop=True)

    # Now aggregate per segment: need unique segment key across groups
    df_segments['segment_key'] = df_segments.apply(lambda r: (r['cmdb_id'], r['kpi_name'], r['anomaly_type'], int(r['segment_id'])), axis=1)

    seg_agg = df_segments.groupby(['cmdb_id', 'kpi_name', 'anomaly_type', 'segment_id'], as_index=False).agg(
        start_timestamp_epoch = ('timestamp', 'min'),
        end_timestamp_epoch   = ('timestamp', 'max'),
        length_minutes = ('timestamp', 'count'),  # contiguous minutes -> count equals inclusive minutes
        max_value_in_segment = ('value', 'max'),
        min_value_in_segment = ('value', 'min'),
        p5_value = ('p5_value', 'first'),
        p95_value = ('p95_value', 'first')
    )

    # Compute max_deviation_pct relative to p95 for 'high' or p5 for 'low'
    def compute_max_dev(row):
        if row['anomaly_type'] == 'high':
            base = row['p95_value']
            if pd.isna(base) or base == 0:
                return np.nan
            return (row['max_value_in_segment'] - base) / base * 100.0
        else:
            base = row['p5_value']
            if pd.isna(base) or base == 0:
                return np.nan
            return (base - row['min_value_in_segment']) / base * 100.0

    seg_agg['max_deviation_pct'] = seg_agg.apply(compute_max_dev, axis=1)

    # Convert epoch to Asia/Shanghai timezone-aware datetimes
    seg_agg['start_timestamp'] = pd.to_datetime(seg_agg['start_timestamp_epoch'], unit='s', utc=True).dt.tz_convert(tz)
    seg_agg['end_timestamp']   = pd.to_datetime(seg_agg['end_timestamp_epoch'], unit='s', utc=True).dt.tz_convert(tz)

    # Select and order requested columns
    osb_fault_segments = seg_agg[
        ['cmdb_id', 'kpi_name', 'anomaly_type', 'start_timestamp', 'end_timestamp',
         'length_minutes', 'max_value_in_segment', 'min_value_in_segment', 'max_deviation_pct']
    ].sort_values('start_timestamp', ascending=True).head(20).reset_index(drop=True)

    # Round numeric columns for compactness
    osb_fault_segments['max_value_in_segment'] = osb_fault_segments['max_value_in_segment'].round(6)
    osb_fault_segments['min_value_in_segment'] = osb_fault_segments['min_value_in_segment'].round(6)
    osb_fault_segments['max_deviation_pct'] = osb_fault_segments['max_deviation_pct'].round(3)
else:
    # No anomalies found -> empty DataFrame with desired columns
    osb_fault_segments = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'anomaly_type', 'start_timestamp', 'end_timestamp',
        'length_minutes', 'max_value_in_segment', 'min_value_in_segment', 'max_deviation_pct'
    ])

# Display the compact result variable (per rule: use variable name for output)
osb_fault_segments

```
Out[4]:
```
```python
# Summarize osb_001 app.* fault segments in plain English using existing kernel variable osb_fault_segments.

try:
    df = osb_fault_segments.copy()
except NameError:
    df = None

if df is None or df.empty:
    summary = "No fault segments found for osb_001 app.* KPIs in the incident window."
else:
    n_segments = int(df.shape[0])
    n_high = int((df['anomaly_type'] == 'high').sum())
    n_low = int((df['anomaly_type'] == 'low').sum())
    window_start = df['start_timestamp'].min()
    window_end = df['end_timestamp'].max()
    # Top segments by start time (first 6 for concise overview)
    top = df.sort_values('start_timestamp').head(6)
    lines = [
        f"Summary of fault segments for cmdb_id=osb_001, KPIs app.* within 2020-05-28 03:30–04:00+08:00:",
        f"- Total fault segments found: {n_segments} (high={n_high}, low={n_low}).",
        f"- Observed time span among segments: {window_start} to {window_end}.",
        "- Key segments (start → end; type; length_minutes; max/min value; max_deviation_pct):"
    ]
    for _, r in top.iterrows():
        lines.append(
            f"  • {r['kpi_name']}: {r['start_timestamp']} → {r['end_timestamp']}; "
            f"type={r['anomaly_type']}; length={int(r['length_minutes'])} min; "
            f"max={r['max_value_in_segment']}; min={r['min_value_in_segment']}; "
            f"max_dev%={r['max_deviation_pct']}"
        )
    # Highlight notable findings
    # 1) Large high deviation on app.avg_time segment
    high_avg = df[(df['kpi_name']=='app.avg_time') & (df['anomaly_type']=='high')]
    if not high_avg.empty:
        h = high_avg.sort_values('start_timestamp').iloc[0]
        lines.append(
            f"- Notable: app.avg_time had a high anomaly from {h['start_timestamp']} to {h['end_timestamp']} "
            f"({int(h['length_minutes'])} min) with very large max_deviation_pct={h['max_deviation_pct']}%."
        )
    # 2) Concurrent low anomalies on app.num and app.succee_num
    concurrent = df[
        (df['start_timestamp'] == pd.Timestamp('2020-05-28 03:48:00+08:00'))
        & (df['kpi_name'].isin(['app.num','app.succee_num']))
    ]
    if not concurrent.empty:
        lines.append(
            "- Concurrent anomaly: at 03:48–03:50 there were 3-minute low segments on app.num and app.succee_num "
            "(and a high on app.avg_time), indicating a correlated event affecting counts and latency."
        )
    lines.append("- End of summary.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         kpi_name anomaly_type           start_timestamp             end_timestamp  length_minutes  max_value_in_segment  min_value_in_segment  max_deviation_pct
0  osb_001     app.avg_time          low 2020-05-28 03:43:00+08:00 2020-05-28 03:43:00+08:00               1              0.000429              0.000429              8.513
1  osb_001     app.avg_time          low 2020-05-28 03:46:00+08:00 2020-05-28 03:46:00+08:00               1              0.000438              0.000438              6.614
2  osb_001  app.succee_rate          low 2020-05-28 03:47:00+08:00 2020-05-28 03:47:00+08:00               1              0.985100              0.985100              1.184
3  osb_001     app.avg_time         high 2020-05-28 03:48:00+08:00 2020-05-28 03:50:00+08:00               3              0.067862              0.009725           1107.506
4  osb_001          app.num          low 2020-05-28 03:48:00+08:00 2020-05-28 03:50:00+08:00               3             75.000000             66.000000             23.256
5  osb_001   app.succee_num          low 2020-05-28 03:48:00+08:00 2020-05-28 03:50:00+08:00               3             74.000000             66.000000             20.958
6  osb_001  app.succee_rate          low 2020-05-28 03:49:00+08:00 2020-05-28 03:50:00+08:00               2              0.996050              0.993300              0.361
7  osb_001     app.avg_time          low 2020-05-28 03:57:00+08:00 2020-05-28 03:57:00+08:00               1              0.000460              0.000460              1.899
8  osb_001          app.num         high 2020-05-28 04:00:00+08:00 2020-05-28 04:00:00+08:00               1            966.000000            966.000000            107.742
9  osb_001   app.succee_num         high 2020-05-28 04:00:00+08:00 2020-05-28 04:00:00+08:00               1            963.000000            963.000000            107.097```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse final_result from kernel which contains metric anomaly aggregates
try:
    final_result
except NameError:
    # If final_result is missing, raise a clear error
    raise NameError("final_result not found in kernel. Ensure previous steps executed and final_result exists.")

tz = pytz.timezone('Asia/Shanghai')

# Filter rows where cmdb_id starts with 'db_'
db_anoms = final_result[final_result['cmdb_id'].str.startswith('db_')].copy()

# Ensure earliest_anomaly_ts is timezone-aware in Asia/Shanghai (it should be already)
if not pd.api.types.is_datetime64tz_dtype(db_anoms['earliest_anomaly_ts']):
    db_anoms['earliest_anomaly_ts'] = pd.to_datetime(db_anoms['earliest_anomaly_ts'], unit='s', utc=True).dt.tz_convert(tz)

# Prepare numeric deviation columns and compute max_deviation_pct treating NULLs appropriately
# Convert columns to numeric (they may already be), keep NaN where appropriate
db_anoms['max_high_deviation_pct'] = pd.to_numeric(db_anoms['max_high_deviation_pct'], errors='coerce')
db_anoms['max_low_deviation_pct']  = pd.to_numeric(db_anoms['max_low_deviation_pct'], errors='coerce')

# Compute max_deviation_pct: the maximum of the two, but if one is NaN use the other; if both NaN => NaN
db_anoms['max_deviation_pct'] = db_anoms[['max_high_deviation_pct', 'max_low_deviation_pct']].max(axis=1, skipna=True)

# Round deviation percentages for compact display
db_anoms['max_high_deviation_pct'] = db_anoms['max_high_deviation_pct'].round(3)
db_anoms['max_low_deviation_pct']  = db_anoms['max_low_deviation_pct'].round(3)
db_anoms['max_deviation_pct']      = db_anoms['max_deviation_pct'].round(3)

# Select requested columns and sort by max_deviation_pct desc, earliest_anomaly_ts asc
cols = ['cmdb_id', 'kpi_name', 'p5_value', 'p95_value',
        'anomaly_high_count', 'anomaly_low_count', 'earliest_anomaly_ts',
        'max_high_deviation_pct', 'max_low_deviation_pct', 'max_deviation_pct']

db_anoms_result = db_anoms[cols].sort_values(
    by=['max_deviation_pct', 'earliest_anomaly_ts'],
    ascending=[False, True],
    na_position='last'
).head(20)

# Display compact result variable
db_anoms_result

```
Out[5]:
```
```python
# Summarize DB anomaly findings in plain English using existing variable db_anoms_result.
# The summary string will state whether any DB component shows >50% deviation and list earliest anomaly times.

try:
    df = db_anoms_result.copy()
except NameError:
    df = None

if df is None or df.empty:
    summary = "No DB anomalies found in the previously computed results."
else:
    # Count affected DB components and list key details
    n_db = df.shape[0]
    # Find if any exceed 50% deviation
    exceed50 = df[df['max_deviation_pct'] > 50.0]
    exceed_flag = exceed50.shape[0] > 0

    # Prepare earliest anomaly times per row
    rows = []
    for _, r in df.sort_values('earliest_anomaly_ts').iterrows():
        rows.append(f"{r['cmdb_id']} ({r['kpi_name']}): max_dev%={r['max_deviation_pct']}%, earliest={r['earliest_anomaly_ts']}")

    summary_lines = [
        f"DB anomaly summary: {n_db} DB component-KPI series showed anomalies in the incident window.",
        f"- KPI involved: all rows are for CPU_free_pct.",
        f"- Any DB with >50% deviation? {'Yes' if exceed_flag else 'No (none exceed 50%)'}.",
        "- Affected DB components and earliest anomalies (sorted by time):"
    ]
    summary_lines += [f"  - {line}" for line in rows]
    summary_lines.append("- Conclusion: no DB component exhibits a very large (>50%) deviation; the largest observed max_deviation_pct values are 7.615%, 1.593%, and 0.842% (earliest anomalies at 03:31, 03:32, and 03:37 +08:00 respectively).")

    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id      kpi_name   p5_value  p95_value  anomaly_high_count  anomaly_low_count       earliest_anomaly_ts  max_high_deviation_pct  max_low_deviation_pct  max_deviation_pct
5    db_001  CPU_free_pct  97.143088  98.997902                   3                  2 2020-05-28 03:32:00+08:00                   0.101                  7.615              7.615
91   db_003  CPU_free_pct  97.799505  99.348405                   3                  1 2020-05-28 03:31:00+08:00                   0.159                  1.593              1.593
312  db_008  CPU_free_pct  96.295133  97.638150                   2                  1 2020-05-28 03:37:00+08:00                   0.081                  0.842              0.842```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables from kernel: final_result (metrics) and final_trace_anomalies (traces)
try:
    metrics_df = final_result.copy()
except NameError:
    raise NameError("final_result not found in kernel")

try:
    traces_df = final_trace_anomalies.copy()
except NameError:
    raise NameError("final_trace_anomalies not found in kernel")

tz = pytz.timezone('Asia/Shanghai')

# Ensure earliest_anomaly_ts are timezone-aware and comparable
def ensure_tz(series):
    if series.dtype == 'O':
        # try to convert object to datetimes
        series = pd.to_datetime(series, utc=True, errors='coerce').dt.tz_convert(tz)
    elif not pd.api.types.is_datetime64tz_dtype(series):
        series = pd.to_datetime(series, unit='s', utc=True, errors='coerce').dt.tz_convert(tz)
    else:
        series = series.dt.tz_convert(tz)
    return series

metrics_df['earliest_anomaly_ts'] = ensure_tz(metrics_df['earliest_anomaly_ts'])
traces_df['earliest_anomaly_ts'] = ensure_tz(traces_df['earliest_anomaly_ts'])

# Normalize deviation columns to numeric
metrics_df['max_high_deviation_pct'] = pd.to_numeric(metrics_df.get('max_high_deviation_pct'), errors='coerce')
metrics_df['max_low_deviation_pct']  = pd.to_numeric(metrics_df.get('max_low_deviation_pct'), errors='coerce')
traces_df['max_high_deviation_pct']  = pd.to_numeric(traces_df.get('max_high_deviation_pct'), errors='coerce')
traces_df['max_low_deviation_pct']   = pd.to_numeric(traces_df.get('max_low_deviation_pct'), errors='coerce')

# Threshold for high impact
THRESH = 50.0

# Collect metric events where either deviation > THRESH
metric_high = metrics_df[metrics_df['max_high_deviation_pct'] > THRESH].copy()
metric_low  = metrics_df[metrics_df['max_low_deviation_pct']  > THRESH].copy()

metric_rows = []
if not metric_high.empty:
    mh = metric_high.assign(
        source_type='metric',
        cmdb_id=metric_high['cmdb_id'],
        trace_name=np.nan,
        kpi_or_trace=metric_high['kpi_name'],
        which_deviation='high',
        max_deviation_pct=metric_high['max_high_deviation_pct'],
        earliest_anomaly_ts=metric_high['earliest_anomaly_ts']
    )[ ['source_type','cmdb_id','trace_name','kpi_or_trace','which_deviation','max_deviation_pct','earliest_anomaly_ts'] ]
    metric_rows.append(mh)

if not metric_low.empty:
    ml = metric_low.assign(
        source_type='metric',
        cmdb_id=metric_low['cmdb_id'],
        trace_name=np.nan,
        kpi_or_trace=metric_low['kpi_name'],
        which_deviation='low',
        max_deviation_pct=metric_low['max_low_deviation_pct'],
        earliest_anomaly_ts=metric_low['earliest_anomaly_ts']
    )[ ['source_type','cmdb_id','trace_name','kpi_or_trace','which_deviation','max_deviation_pct','earliest_anomaly_ts'] ]
    metric_rows.append(ml)

metrics_events = pd.concat(metric_rows, ignore_index=True) if metric_rows else pd.DataFrame(columns=['source_type','cmdb_id','trace_name','kpi_or_trace','which_deviation','max_deviation_pct','earliest_anomaly_ts'])

# Collect trace events where either deviation > THRESH
trace_high = traces_df[traces_df['max_high_deviation_pct'] > THRESH].copy()
trace_low  = traces_df[traces_df['max_low_deviation_pct']  > THRESH].copy()

trace_rows = []
if not trace_high.empty:
    th = trace_high.assign(
        source_type='trace',
        cmdb_id=np.nan,
        trace_name=trace_high['trace_name'],
        kpi_or_trace=trace_high['trace_name'],
        which_deviation='high',
        max_deviation_pct=trace_high['max_high_deviation_pct'],
        earliest_anomaly_ts=trace_high['earliest_anomaly_ts']
    )[ ['source_type','cmdb_id','trace_name','kpi_or_trace','which_deviation','max_deviation_pct','earliest_anomaly_ts'] ]
    trace_rows.append(th)

if not trace_low.empty:
    tl = trace_low.assign(
        source_type='trace',
        cmdb_id=np.nan,
        trace_name=trace_low['trace_name'],
        kpi_or_trace=trace_low['trace_name'],
        which_deviation='low',
        max_deviation_pct=trace_low['max_low_deviation_pct'],
        earliest_anomaly_ts=trace_low['earliest_anomaly_ts']
    )[ ['source_type','cmdb_id','trace_name','kpi_or_trace','which_deviation','max_deviation_pct','earliest_anomaly_ts'] ]
    trace_rows.append(tl)

traces_events = pd.concat(trace_rows, ignore_index=True) if trace_rows else pd.DataFrame(columns=['source_type','cmdb_id','trace_name','kpi_or_trace','which_deviation','max_deviation_pct','earliest_anomaly_ts'])

# Combine metric and trace events
high_impact_events = pd.concat([metrics_events, traces_events], ignore_index=True)

# Ensure correct dtypes and formatting
if not high_impact_events.empty:
    high_impact_events['max_deviation_pct'] = pd.to_numeric(high_impact_events['max_deviation_pct'], errors='coerce').round(3)
    high_impact_events['earliest_anomaly_ts'] = ensure_tz(high_impact_events['earliest_anomaly_ts'])
    # Sort by earliest_anomaly_ts ascending and limit to top 20
    high_impact_events = high_impact_events.sort_values('earliest_anomaly_ts', ascending=True).head(20).reset_index(drop=True)

# Return the compact table (variable as output)
high_impact_events

```
Out[6]:
```
```python
# Summarize high-impact events (deviation > 50%) found in high_impact_events variable.
# Reuse existing high_impact_events from kernel.

try:
    df = high_impact_events.copy()
except NameError:
    df = None

if df is None or df.empty:
    summary = "No high-impact events (deviation > 50%) found."
else:
    # Counts
    total_events = int(df.shape[0])
    n_metric = int((df['source_type'] == 'metric').sum())
    n_trace  = int((df['source_type'] == 'trace').sum())
    # Earliest anomaly overall
    earliest_overall = df['earliest_anomaly_ts'].min()
    # Top 6 by max_deviation_pct (severity), present key fields
    top_severe = df.sort_values('max_deviation_pct', ascending=False).head(6)
    top_lines = []
    for _, r in top_severe.iterrows():
        id_field = (r['cmdb_id'] if pd.notna(r['cmdb_id']) else r['trace_name'])
        top_lines.append(
            f"{id_field} / {r['kpi_or_trace']} ({r['which_deviation']}): "
            f"{r['max_deviation_pct']}% at {r['earliest_anomaly_ts']}"
        )
    # Compose summary
    summary_lines = [
        f"High-impact anomaly summary: {total_events} events with deviation >50% (metrics={n_metric}, traces={n_trace}).",
        f"Earliest high-impact anomaly observed at: {earliest_overall}.",
        "Top severe events (by max_deviation_pct):"
    ]
    summary_lines += [f"  - {ln}" for ln in top_lines]
    # Note notable metric findings if present
    metrics_notable = df[df['source_type']=='metric']
    if not metrics_notable.empty:
        mlist = ", ".join(sorted(metrics_notable['cmdb_id'].unique()))
        summary_lines.append(f"Notable affected metrics components: {mlist}.")
    summary_lines.append("End of summary.")
    summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

source_type  cmdb_id                          trace_name                        kpi_or_trace which_deviation  max_deviation_pct       earliest_anomaly_ts
0        trace      NaN      trace.self.local.duration_mean      trace.self.local.duration_mean            high           2012.964 2020-05-28 03:33:00+08:00
1        trace      NaN       trace.self.local.duration_p95       trace.self.local.duration_p95            high           1563.711 2020-05-28 03:33:00+08:00
2        trace      NaN        trace.self.csf.duration_mean        trace.self.csf.duration_mean            high            149.923 2020-05-28 03:33:00+08:00
3        trace      NaN         trace.self.csf.duration_p95         trace.self.csf.duration_p95            high            203.637 2020-05-28 03:35:00+08:00
4       metric   os_012                                 NaN                        Disk_io_util            high             89.034 2020-05-28 03:37:00+08:00
5        trace      NaN       trace.self.jdbc.duration_mean       trace.self.jdbc.duration_mean            high           3611.324 2020-05-28 03:37:00+08:00
6        trace      NaN        trace.self.jdbc.duration_p95        trace.self.jdbc.duration_p95            high          11299.387 2020-05-28 03:39:00+08:00
7       metric  osb_001                                 NaN                        app.avg_time            high           1107.506 2020-05-28 03:43:00+08:00
8        trace      NaN  trace.self.flyremote.duration_mean  trace.self.flyremote.duration_mean            high           1614.897 2020-05-28 03:43:00+08:00
9        trace      NaN   trace.self.flyremote.duration_p95   trace.self.flyremote.duration_p95            high           1518.731 2020-05-28 03:47:00+08:00
10      metric  osb_001                                 NaN                             app.num            high            107.742 2020-05-28 03:48:00+08:00
11      metric  osb_001                                 NaN                      app.succee_num            high            107.097 2020-05-28 03:48:00+08:00```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: df_trace, traces_summary
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

try:
    traces_summary
except NameError:
    # If traces_summary missing, compute global p95 per trace_name (full series) as fallback
    traces_summary = df_trace.groupby('trace_name', as_index=False).agg(
        p95_value=('value', lambda x: np.percentile(x, 95))
    )

tz = pytz.timezone('Asia/Shanghai')

# Ensure traces_summary has p95_value column
if 'p95_value' not in traces_summary.columns:
    traces_summary = traces_summary.rename(columns={traces_summary.columns[-1]: 'p95_value'})

# 1) Filter trace.csv to the incident window (inclusive)
start_ts = int(pd.Timestamp('2020-05-28 03:30:00', tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp('2020-05-28 04:00:00', tz=tz).tz_convert('UTC').timestamp())

df_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 2) Attach global p95_value per trace_name (do NOT recompute from window)
p95_df = traces_summary[['trace_name', 'p95_value']].drop_duplicates()
df_window = df_window.merge(p95_df, on='trace_name', how='left')

# Identify high values using the global p95_value
df_window['is_high'] = df_window['p95_value'].notna() & (df_window['value'] > df_window['p95_value'])

# Keep only high rows to aggregate count and earliest timestamp per (trace_name, cmdb_id)
df_high = df_window[df_window['is_high']].copy()

if not df_high.empty:
    grouped = df_high.groupby(['trace_name', 'cmdb_id'], as_index=False).agg(
        count_high=('is_high', 'sum'),
        earliest_high_ts_epoch=('timestamp', 'min'),
        p95_value=('p95_value', 'first')
    )
    # Convert earliest timestamp to Asia/Shanghai timezone-aware datetimes
    grouped['earliest_high_ts'] = pd.to_datetime(grouped['earliest_high_ts_epoch'], unit='s', utc=True).dt.tz_convert(tz)
    # Select and order columns
    trace_anoms_by_component = grouped[['trace_name', 'cmdb_id', 'count_high', 'earliest_high_ts', 'p95_value']] \
        .sort_values('earliest_high_ts', ascending=True).head(20).reset_index(drop=True)
else:
    # No high anomalies detected in window
    trace_anoms_by_component = pd.DataFrame(columns=['trace_name', 'cmdb_id', 'count_high', 'earliest_high_ts', 'p95_value'])

# Return the compact table variable
trace_anoms_by_component

```
Out[7]:
```
```python
# Summarize trace_anoms_by_component in plain English and display as a single string variable.
# Reuse the existing variable trace_anoms_by_component from the kernel.

try:
    df = trace_anoms_by_component.copy()
except NameError:
    df = None

if df is None or df.empty:
    summary = "No high trace anomalies found in the incident window."
else:
    # Key observations
    earliest_row = df.sort_values('earliest_high_ts').iloc[0]
    # Find components with many early anomalies (within first minute cluster 03:36)
    early_cluster = df[df['earliest_high_ts'] <= pd.Timestamp('2020-05-28 03:36:00+08:00')]
    # Identify the most frequent offender overall (by count_high)
    top_offender = df.sort_values('count_high', ascending=False).iloc[0]

    lines = [
        "Trace anomaly summary (high > global P95) for 2020-05-28 03:30–04:00+08:00:",
        "",
        f"- Earliest high anomaly observed: {earliest_row['trace_name']} on {earliest_row['cmdb_id']} at {earliest_row['earliest_high_ts']} (count_high={int(earliest_row['count_high'])}, p95={earliest_row['p95_value']}).",
        "",
        "- Notable early cluster (~03:36+08:00):",
        "  • docker_003 shows multiple high-impact traces at ~03:36 (e.g., trace.self.jdbc.row_count count_high=6, trace.self.local.row_count count_high=5, trace.self.csf.row_count count_high=3).",
        "  • docker_008 and docker_007 also show high events around 03:36–03:41 for remoteprocess traces.",
        "",
        f"- Most frequent single offender by count: {top_offender['trace_name']} on {top_offender['cmdb_id']} with count_high={int(top_offender['count_high'])} (earliest {top_offender['earliest_high_ts']}).",
        "",
        "- Interpretation / guidance:",
        "  • The very first high event came from docker_007 at 03:35, but docker_003 quickly exhibits the largest number of high-impact trace anomalies starting at ~03:36.",
        "  • This pattern makes docker_003 a strong candidate for a downstream faulty service contributing to the incident; docker_002 and docker_006 also show concentrated high anomalies later (~03:47).",
        "",
        "Recommendation: prioritize investigating docker_003 (JDBC/local/csf traces) first, then inspect docker_007 and docker_002/docker_006 as secondary suspects."
    ]
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

trace_name     cmdb_id  count_high          earliest_high_ts    p95_value
0    trace.from_docker_001.remoteprocess.duration_p95  docker_007           2 2020-05-28 03:35:00+08:00     0.056372
1                           trace.self.jdbc.row_count  docker_003           6 2020-05-28 03:36:00+08:00  1826.050000
2   trace.from_docker_001.remoteprocess.duration_mean  docker_008           2 2020-05-28 03:36:00+08:00     0.047048
3    trace.from_docker_001.remoteprocess.duration_p95  docker_008           1 2020-05-28 03:36:00+08:00     0.056372
4                          trace.self.local.row_count  docker_003           5 2020-05-28 03:36:00+08:00  1596.000000
5                            trace.self.csf.row_count  docker_003           3 2020-05-28 03:36:00+08:00   468.000000
6                          trace.self.local.row_count  docker_002           4 2020-05-28 03:38:00+08:00  1596.000000
7                           trace.self.jdbc.row_count  docker_002           4 2020-05-28 03:38:00+08:00  1826.050000
8                        trace.self.jdbc.duration_p95  docker_006           6 2020-05-28 03:39:00+08:00     0.008150
9                        trace.self.jdbc.duration_p95  docker_005           1 2020-05-28 03:39:00+08:00     0.008150
10  trace.from_docker_001.remoteprocess.duration_mean  docker_007           1 2020-05-28 03:41:00+08:00     0.047048
11      trace.from_docker_001.remoteprocess.row_count  docker_008           2 2020-05-28 03:46:00+08:00   230.150000
12                          trace.self.csf.error_rate  docker_004           2 2020-05-28 03:47:00+08:00     0.000000
13                       trace.self.csf.duration_mean      os_021           4 2020-05-28 03:47:00+08:00     2.853495
14                       trace.self.jdbc.duration_p95  docker_002           5 2020-05-28 03:47:00+08:00     0.008150
15                     trace.self.local.duration_mean  docker_002           7 2020-05-28 03:47:00+08:00     0.019569
16                      trace.self.jdbc.duration_mean  docker_006           5 2020-05-28 03:47:00+08:00     0.007268
17                      trace.self.jdbc.duration_mean  docker_002           5 2020-05-28 03:47:00+08:00     0.007268
18                          trace.self.csf.error_rate  docker_003           2 2020-05-28 03:47:00+08:00     0.000000
19                       trace.self.csf.duration_mean      os_022           5 2020-05-28 03:47:00+08:00     2.853495```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz

# Reuse existing kernel variable trace_anoms_by_component
try:
    trace_anoms_by_component
except NameError:
    trace_anoms_by_component = pd.DataFrame(columns=['trace_name','cmdb_id','count_high','earliest_high_ts','p95_value'])

tz = pytz.timezone('Asia/Shanghai')

# Ensure earliest_high_ts is timezone-aware Asia/Shanghai
def ensure_tz_series(s):
    if pd.api.types.is_datetime64tz_dtype(s):
        return s.dt.tz_convert(tz)
    else:
        return pd.to_datetime(s, utc=True, errors='coerce').dt.tz_convert(tz)

trace_anoms_by_component['earliest_high_ts'] = ensure_tz_series(trace_anoms_by_component['earliest_high_ts'])

# Aggregate by cmdb_id
trace_candidate_summary = (
    trace_anoms_by_component
    .groupby('cmdb_id', as_index=False)
    .agg(
        total_high_count = ('count_high', 'sum'),
        earliest_high_ts  = ('earliest_high_ts', 'min'),
        distinct_traces_count = ('trace_name', 'nunique')
    )
)

# Sort and limit to top 10 as requested
trace_candidate_summary = trace_candidate_summary.sort_values(
    by=['total_high_count', 'earliest_high_ts'],
    ascending=[False, True]
).head(10).reset_index(drop=True)

# Display compact result variable
trace_candidate_summary

```
Out[8]:
```
```python
# Summarize trace candidate containers from trace_candidate_summary in plain English.
# Reuse the existing variable trace_candidate_summary from the kernel and produce a compact summary string.

try:
    df = trace_candidate_summary.copy()
except NameError:
    df = pd.DataFrame()

if df.empty:
    summary = "No trace candidate data available."
else:
    # Build top entries summary
    top_entries = []
    for _, r in df.head(5).iterrows():
        ts = r['earliest_high_ts']
        ts_str = str(ts) if pd.notna(ts) else "N/A"
        top_entries.append(f"{r['cmdb_id']} (total_high_count={int(r['total_high_count'])}, distinct_traces={int(r['distinct_traces_count'])}, earliest={ts_str})")
    top_line = "; ".join(top_entries)

    # Earliest overall high anomaly (may be a container with fewer events)
    earliest_row = df.sort_values('earliest_high_ts', ascending=True).iloc[0]
    earliest_note = f"Earliest high anomaly observed on {earliest_row['cmdb_id']} at {earliest_row['earliest_high_ts']} (count_high={int(earliest_row['total_high_count'])})."

    # Recommendation
    recommendation = "Priority: investigate docker_002 and docker_003 first (highest counts and multiple distinct trace types). Also check docker_007 for the very first observed high at ~03:35."

    summary = (
        "Trace candidate summary (top containers by high-impact trace events): "
        f"{top_line}. "
        f"{earliest_note} "
        f"{recommendation}"
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_high_count          earliest_high_ts  distinct_traces_count
0  docker_002                25 2020-05-28 03:38:00+08:00                      5
1  docker_003                16 2020-05-28 03:36:00+08:00                      4
2  docker_006                11 2020-05-28 03:39:00+08:00                      2
3  docker_008                 5 2020-05-28 03:36:00+08:00                      3
4      os_022                 5 2020-05-28 03:47:00+08:00                      1
5      os_021                 4 2020-05-28 03:47:00+08:00                      1
6  docker_007                 3 2020-05-28 03:35:00+08:00                      2
7  docker_004                 2 2020-05-28 03:47:00+08:00                      1
8  docker_005                 1 2020-05-28 03:39:00+08:00                      1```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# Reuse existing kernel variables: trace_anoms_by_component, osb_fault_segments
try:
    trace_anoms_by_component
except NameError:
    trace_anoms_by_component = pd.DataFrame(columns=['trace_name','cmdb_id','count_high','earliest_high_ts','p95_value'])

try:
    osb_fault_segments
except NameError:
    osb_fault_segments = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'anomaly_type', 'start_timestamp', 'end_timestamp',
        'length_minutes', 'max_value_in_segment', 'min_value_in_segment', 'max_deviation_pct'
    ])

tz = pytz.timezone('Asia/Shanghai')

def ensure_tz_series(s):
    # ensure series is timezone-aware in Asia/Shanghai
    if pd.api.types.is_datetime64tz_dtype(s):
        return s.dt.tz_convert(tz)
    else:
        return pd.to_datetime(s, utc=True, errors='coerce').dt.tz_convert(tz)

# 1) Confirm trace evidence for selected containers
containers = ['docker_003', 'docker_002', 'docker_007']
confirm_trace_evidence = trace_anoms_by_component[
    trace_anoms_by_component['cmdb_id'].isin(containers)
].copy()

if not confirm_trace_evidence.empty:
    confirm_trace_evidence['earliest_high_ts'] = ensure_tz_series(confirm_trace_evidence['earliest_high_ts'])
    confirm_trace_evidence = confirm_trace_evidence[
        ['trace_name', 'cmdb_id', 'count_high', 'earliest_high_ts', 'p95_value']
    ].sort_values('earliest_high_ts', ascending=True).head(20).reset_index(drop=True)
else:
    confirm_trace_evidence = confirm_trace_evidence[[
        'trace_name', 'cmdb_id', 'count_high', 'earliest_high_ts', 'p95_value'
    ]]

# 2) Confirm osb_001 fault segments for app.* KPIs
kpis = ['app.avg_time','app.num','app.succee_num','app.succee_rate']
confirm_osb_segments = osb_fault_segments[
    (osb_fault_segments['cmdb_id'] == 'osb_001') &
    (osb_fault_segments['kpi_name'].isin(kpis))
].copy()

if not confirm_osb_segments.empty:
    # ensure timezone and select requested columns
    confirm_osb_segments['start_timestamp'] = ensure_tz_series(confirm_osb_segments['start_timestamp'])
    confirm_osb_segments['end_timestamp'] = ensure_tz_series(confirm_osb_segments['end_timestamp'])
    confirm_osb_segments['max_deviation_pct'] = pd.to_numeric(confirm_osb_segments['max_deviation_pct'], errors='coerce').round(3)
    confirm_osb_segments = confirm_osb_segments[
        ['kpi_name', 'anomaly_type', 'start_timestamp', 'end_timestamp', 'length_minutes', 'max_deviation_pct']
    ].sort_values('start_timestamp', ascending=True).head(20).reset_index(drop=True)
else:
    confirm_osb_segments = confirm_osb_segments[[
        'kpi_name', 'anomaly_type', 'start_timestamp', 'end_timestamp', 'length_minutes', 'max_deviation_pct'
    ]]

# Return the two compact tables (variables as outputs)
confirm_trace_evidence, confirm_osb_segments

```
Out[9]:
```
```python
# Summarize confirming evidence from the two compact tables in plain English.
# Reuse existing variables: confirm_trace_evidence, confirm_osb_segments

try:
    trace_tbl = confirm_trace_evidence.copy()
    osb_tbl = confirm_osb_segments.copy()
except NameError:
    trace_tbl = pd.DataFrame()
    osb_tbl = pd.DataFrame()

# Build concise summary lines based on the small tables
lines = []
lines.append("Timeline & confirming evidence (concise):")

# Earliest trace high events
if not trace_tbl.empty:
    first = trace_tbl.sort_values('earliest_high_ts').iloc[0]
    lines.append(f"- Earliest high trace event: {first['trace_name']} on {first['cmdb_id']} at {first['earliest_high_ts']}.")
    # key early cluster: docker_003 entries at ~03:36
    cluster_36 = trace_tbl[trace_tbl['earliest_high_ts'].astype(str).str.contains("03:36:00")]
    if not cluster_36.empty:
        lines.append("- Around 03:36+08:00 there is a cluster on docker_003: multiple trace metrics (jdbc.row_count, local.row_count, csf.row_count) showed high counts.")
    # docker_002 heavier later
    later_docker_002 = trace_tbl[trace_tbl['cmdb_id']=='docker_002']
    if not later_docker_002.empty:
        earliest_002 = later_docker_002['earliest_high_ts'].min()
        lines.append(f"- docker_002 shows concentrated high anomalies starting at {earliest_002}, and affects several trace types (multiple distinct traces).")
    # docker_007 earliest but fewer traces
    if 'docker_007' in trace_tbl['cmdb_id'].values:
        lines.append("- docker_007 had the very first observed high at 03:35 but with fewer distinct trace types than docker_003/docker_002.")

# OSB app metrics evidence
if not osb_tbl.empty:
    lines.append("- Application-side segments (osb_001 app.*):")
    # Highlight the major latency spike
    high_latency = osb_tbl[(osb_tbl['kpi_name']=='app.avg_time') & (osb_tbl['anomaly_type']=='high')]
    if not high_latency.empty:
        hl = high_latency.sort_values('start_timestamp').iloc[0]
        lines.append(f"  • app.avg_time had a severe HIGH segment {hl['start_timestamp']} → {hl['end_timestamp']} (length {int(hl['length_minutes'])} min) with max_deviation_pct={hl['max_deviation_pct']}%.")
    # Concurrent count drops
    counts_low = osb_tbl[osb_tbl['kpi_name'].isin(['app.num','app.succee_num','app.succee_rate'])]
    if not counts_low.empty:
        # find concurrent segment around 03:48
        concurrent = counts_low[(counts_low['start_timestamp'] >= pd.Timestamp("2020-05-28 03:48:00+08:00")) & (counts_low['start_timestamp'] <= pd.Timestamp("2020-05-28 03:50:00+08:00"))]
        if not concurrent.empty:
            lines.append("  • At ~03:48–03:50+08:00 there are concurrent LOW segments on app.num and app.succee_num (3-minute drops) while app.avg_time spikes — indicating the app observed latency increase while throughput/success counts dropped.")

# Short conclusion / interpretation
lines.append("Interpretation:")
lines.append("- Traces: the first sign is from docker_007 at 03:35, but docker_003 shows a strong early cluster (~03:36) across multiple trace types (JDBC/local/CSF), and docker_002 shows larger aggregated counts later.")
lines.append("- Application impact: osb_001 experienced a massive latency spike (app.avg_time) at 03:48–03:50 concurrent with drops in app.num and app.succee_num.")
lines.append("- Triage suggestion: investigate docker_003 (early multi-trace anomalies) and docker_002 (high aggregated counts) as likely downstream faulty containers; docker_007 is the earliest single-signal but with fewer traces. The application impact (root-cause manifestation) is observed around 03:48–03:50+08:00 on osb_001.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                                           trace_name     cmdb_id  count_high          earliest_high_ts    p95_value
0    trace.from_docker_001.remoteprocess.duration_p95  docker_007           2 2020-05-28 03:35:00+08:00     0.056372
1                           trace.self.jdbc.row_count  docker_003           6 2020-05-28 03:36:00+08:00  1826.050000
2                          trace.self.local.row_count  docker_003           5 2020-05-28 03:36:00+08:00  1596.000000
3                            trace.self.csf.row_count  docker_003           3 2020-05-28 03:36:00+08:00   468.000000
4                          trace.self.local.row_count  docker_002           4 2020-05-28 03:38:00+08:00  1596.000000
5                           trace.self.jdbc.row_count  docker_002           4 2020-05-28 03:38:00+08:00  1826.050000
6   trace.from_docker_001.remoteprocess.duration_mean  docker_007           1 2020-05-28 03:41:00+08:00     0.047048
7                        trace.self.jdbc.duration_p95  docker_002           5 2020-05-28 03:47:00+08:00     0.008150
8                      trace.self.local.duration_mean  docker_002           7 2020-05-28 03:47:00+08:00     0.019569
9                       trace.self.jdbc.duration_mean  docker_002           5 2020-05-28 03:47:00+08:00     0.007268
10                          trace.self.csf.error_rate  docker_003           2 2020-05-28 03:47:00+08:00     0.000000,           kpi_name anomaly_type           start_timestamp             end_timestamp  length_minutes  max_deviation_pct
0     app.avg_time          low 2020-05-28 03:43:00+08:00 2020-05-28 03:43:00+08:00               1              8.513
1     app.avg_time          low 2020-05-28 03:46:00+08:00 2020-05-28 03:46:00+08:00               1              6.614
2  app.succee_rate          low 2020-05-28 03:47:00+08:00 2020-05-28 03:47:00+08:00               1              1.184
3     app.avg_time         high 2020-05-28 03:48:00+08:00 2020-05-28 03:50:00+08:00               3           1107.506
4          app.num          low 2020-05-28 03:48:00+08:00 2020-05-28 03:50:00+08:00               3             23.256
5   app.succee_num          low 2020-05-28 03:48:00+08:00 2020-05-28 03:50:00+08:00               3             20.958
6  app.succee_rate          low 2020-05-28 03:49:00+08:00 2020-05-28 03:50:00+08:00               2              0.361
7     app.avg_time          low 2020-05-28 03:57:00+08:00 2020-05-28 03:57:00+08:00               1              1.899
8          app.num         high 2020-05-28 04:00:00+08:00 2020-05-28 04:00:00+08:00               1            107.742
9   app.succee_num         high 2020-05-28 04:00:00+08:00 2020-05-28 04:00:00+08:00               1            107.097)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables where available
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

try:
    metrics_summary
except NameError:
    # Fallback: compute global p95 for all series (allowed since it's global, not from window)
    metrics_summary = df_metric.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        p95_value=('value', lambda x: np.percentile(x, 95))
    )

tz = pytz.timezone('Asia/Shanghai')

# Target component and KPI
cmdb = 'os_012'
kpi = 'Disk_io_util'

# 1) Obtain global p95_value for os_012 Disk_io_util from metrics_summary (do NOT compute from window)
row = metrics_summary[
    (metrics_summary['cmdb_id'] == cmdb) & (metrics_summary['kpi_name'] == kpi)
]
if row.empty:
    # As a safe fallback, compute global p95 from full series for this KPI (still global)
    p95_val = float(df_metric[(df_metric['cmdb_id']==cmdb) & (df_metric['kpi_name']==kpi)]['value'].quantile(0.95))
else:
    # If metrics_summary stores p95_value under that name; otherwise try last numeric column
    if 'p95_value' in row.columns:
        p95_val = float(row['p95_value'].iloc[0])
    else:
        # fallback attempt: take last column as p95
        p95_val = float(row.iloc[0, -1])

# 2) Filter metric.csv to the incident window and target series
start_ts = int(pd.Timestamp('2020-05-28 03:30:00', tz=tz).tz_convert('UTC').timestamp())
end_ts   = int(pd.Timestamp('2020-05-28 04:00:00', tz=tz).tz_convert('UTC').timestamp())

df_window = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'] == cmdb) &
    (df_metric['kpi_name'] == kpi)
].copy()

# 3) Mark points where value > p95_value as 'high'
df_window['anomaly_type'] = np.where(df_window['value'] > p95_val, 'high', pd.NA)

# Keep only high points
df_high = df_window[df_window['anomaly_type'] == 'high'].copy().sort_values('timestamp').reset_index(drop=True)

# 4) Group consecutive 'high' points into contiguous-by-60s segments
if not df_high.empty:
    df_high['gap'] = df_high['timestamp'].diff().fillna(9999)  # big gap for first row
    df_high['new_segment'] = (df_high['gap'] != 60).astype(int)
    df_high['segment_id'] = df_high['new_segment'].cumsum()

    seg_agg = df_high.groupby('segment_id', as_index=False).agg(
        cmdb_id = ('cmdb_id', 'first'),
        kpi_name = ('kpi_name', 'first'),
        anomaly_type = ('anomaly_type', 'first'),
        start_timestamp_epoch = ('timestamp', 'min'),
        end_timestamp_epoch = ('timestamp', 'max'),
        length_minutes = ('timestamp', 'count'),  # contiguous inclusive count
        max_value_in_segment = ('value', 'max'),
        min_value_in_segment = ('value', 'min')
    )

    # Compute max_high_deviation_pct relative to global p95_val
    def comp_dev(max_val):
        if p95_val == 0 or pd.isna(p95_val):
            return np.nan
        return (max_val - p95_val) / p95_val * 100.0

    seg_agg['max_high_deviation_pct'] = seg_agg['max_value_in_segment'].apply(comp_dev)

    # Convert epoch timestamps to Asia/Shanghai timezone-aware datetimes
    seg_agg['start_timestamp'] = pd.to_datetime(seg_agg['start_timestamp_epoch'], unit='s', utc=True).dt.tz_convert(tz)
    seg_agg['end_timestamp']   = pd.to_datetime(seg_agg['end_timestamp_epoch'], unit='s', utc=True).dt.tz_convert(tz)

    # Select requested columns and format numeric rounding
    os_012_disk_faults = seg_agg[
        ['cmdb_id', 'kpi_name', 'anomaly_type', 'start_timestamp', 'end_timestamp',
         'length_minutes', 'max_value_in_segment', 'min_value_in_segment', 'max_high_deviation_pct']
    ].sort_values('start_timestamp', ascending=True).head(20).reset_index(drop=True)

    os_012_disk_faults['max_value_in_segment'] = os_012_disk_faults['max_value_in_segment'].round(6)
    os_012_disk_faults['min_value_in_segment'] = os_012_disk_faults['min_value_in_segment'].round(6)
    os_012_disk_faults['max_high_deviation_pct'] = os_012_disk_faults['max_high_deviation_pct'].round(3)
else:
    # No high anomaly segments found
    os_012_disk_faults = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_type','start_timestamp','end_timestamp',
        'length_minutes','max_value_in_segment','min_value_in_segment','max_high_deviation_pct'
    ])

# Display compact result variable
os_012_disk_faults

```
Out[11]:
```
```python
# Summarize os_012 Disk_io_util fault segments in plain English using existing kernel variable os_012_disk_faults.
try:
    os_012_disk_faults
except NameError:
    os_012_disk_faults = None

if os_012_disk_faults is None or os_012_disk_faults.empty:
    summary = "No Disk_io_util high-impact fault segments found for os_012 in the incident window."
else:
    n_seg = int(os_012_disk_faults.shape[0])
    earliest = os_012_disk_faults['start_timestamp'].min()
    seg_lines = []
    for _, r in os_012_disk_faults.sort_values('start_timestamp').iterrows():
        seg_lines.append(
            f"{r['start_timestamp']} → {r['end_timestamp']}: length={int(r['length_minutes'])} min, "
            f"max={r['max_value_in_segment']}, max_high_deviation_pct={r['max_high_deviation_pct']}%"
        )
    summary = (
        f"os_012 Disk_io_util fault segments (incident window 2020-05-28 03:30–04:00+08:00):\n"
        f"- Total high-impact segments: {n_seg}.\n"
        f"- Earliest segment starts at: {earliest} (Asia/Shanghai).\n"
        f"- Segments:\n  - " + "\n  - ".join(seg_lines) + "\n"
        f"- Interpretation: os_012 experienced short, high-severity Disk_io_util spikes (max ~{os_012_disk_faults['max_high_deviation_pct'].max()}% above global P95). "
        "The earliest confirmed disk fault is at 2020-05-28 03:37+08:00, indicating disk I/O issues on os_012 at that time."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id      kpi_name anomaly_type           start_timestamp             end_timestamp  length_minutes  max_value_in_segment  min_value_in_segment  max_high_deviation_pct
0  os_012  Disk_io_util         high 2020-05-28 03:37:00+08:00 2020-05-28 03:37:00+08:00               1                   2.0                   2.0                  89.034
1  os_012  Disk_io_util         high 2020-05-28 04:00:00+08:00 2020-05-28 04:00:00+08:00               1                   2.0                   2.0                  89.034```
```